20.1 — Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\acer\Desktop\ProgettoTesi"
)

RESULTS_DIR = PROJECT_DIR / "risultati"

CLINICAL_DIR = (
    PROJECT_DIR / "dati clinici"
)

DN4_PATH = (
    CLINICAL_DIR
    / "patient_id_dn4_score.csv"
)


dn4_df = pd.read_csv(
    DN4_PATH
)

print(
    "Shape DN4:",
    dn4_df.shape
)

print(
    "\nColonne:"
)

print(
    dn4_df.columns.tolist()
)

display(
    dn4_df.head()
)

Shape DN4: (243, 12)

Colonne:
['patient_id', 'dn4_1_1', 'dn4_1_2', 'dn4_1_3', 'dn4_2_4', 'dn4_2_5', 'dn4_2_6', 'dn4_2_7', 'dn4_3_8', 'dn4_3_9', 'dn4_4_10', 'dn4_score']


,patient_id,dn4_1_1,dn4_1_2,dn4_1_3,dn4_2_4,dn4_2_5,dn4_2_6,dn4_2_7,dn4_3_8,dn4_3_9,dn4_4_10,dn4_score
0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,7.0
3,4,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,7.0
4,5,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,5.0


20.2 — Metadata multimodali

In [2]:
EMBEDDING_DIR = (
    RESULTS_DIR
    / "embedding_multimodali_turn_level"
)

metadata_df = pd.read_csv(
    EMBEDDING_DIR
    / "metadata_turni_multimodali.csv"
)


print(
    "Turni:",
    len(metadata_df)
)

print(
    "Pazienti:",
    metadata_df[
        "patient_id"
    ].nunique()
)

print(
    "\nTipo patient_id metadata:",
    metadata_df[
        "patient_id"
    ].dtype
)

Turni: 3710
Pazienti: 90

Tipo patient_id metadata: int64


20.3 — Preparazione target DN4

In [3]:
dn4_patient = (
    dn4_df[
        ["patient_id", "dn4_score"]
    ]
    .dropna(
        subset=["patient_id", "dn4_score"]
    )
    .copy()
)

dn4_patient["patient_id"] = (
    dn4_patient["patient_id"]
    .astype(int)
)

# Controllo eventuali duplicati
duplicate_check = (
    dn4_patient
    .groupby("patient_id")["dn4_score"]
    .nunique()
)

conflicting_ids = (
    duplicate_check[
        duplicate_check > 1
    ]
    .index
    .tolist()
)

print(
    "Righe DN4 valide:",
    len(dn4_patient)
)

print(
    "Pazienti DN4 unici:",
    dn4_patient["patient_id"].nunique()
)

print(
    "ID con punteggi DN4 discordanti:",
    len(conflicting_ids)
)

if len(conflicting_ids) > 0:

    print(
        "ATTENZIONE - ID discordanti:",
        conflicting_ids
    )
else:
    # Se un ID fosse ripetuto con lo stesso score,
    # ne manteniamo una sola riga.
    dn4_patient = (
        dn4_patient
        .drop_duplicates(
            subset=["patient_id"]
        )
        .copy()
    )

# Intersezione con i 90 pazienti multimodali
metadata_patient_ids = set(
    metadata_df["patient_id"]
    .astype(int)
    .unique()
)

dn4_patient_ids = set(
    dn4_patient["patient_id"]
    .unique()
)

common_ids = (
    metadata_patient_ids
    & dn4_patient_ids
)

missing_dn4 = sorted(
    metadata_patient_ids
    - dn4_patient_ids
)

print(
    "\nPazienti multimodali:",
    len(metadata_patient_ids)
)

print(
    "Pazienti con DN4 disponibile:",
    len(common_ids)
)

print(
    "Pazienti multimodali senza DN4:",
    len(missing_dn4)
)

print(
    "ID mancanti:",
    missing_dn4
)

Righe DN4 valide: 239
Pazienti DN4 unici: 239
ID con punteggi DN4 discordanti: 0

Pazienti multimodali: 90
Pazienti con DN4 disponibile: 90
Pazienti multimodali senza DN4: 0
ID mancanti: []


20.4 — Merge turni + DN4

In [4]:
turn_dn4 = (
    metadata_df
    .merge(
        dn4_patient,
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )
    .copy()
)

turn_dn4[
    "dn4_binary"
] = (
    turn_dn4[
        "dn4_score"
    ] >= 4
).astype(int)

print(
    "Turni con DN4:",
    len(turn_dn4)
)

print(
    "Pazienti con DN4:",
    turn_dn4[
        "patient_id"
    ].nunique()
)

# Distribuzione a livello PAZIENTE
patient_dn4_labels = (
    turn_dn4[
        [
            "patient_id",
            "dn4_score",
            "dn4_binary"
        ]
    ]
    .drop_duplicates(
        subset=["patient_id"]
    )
)

print(
    "\n===== DISTRIBUZIONE PATIENT-LEVEL ====="
)

display(
    patient_dn4_labels[
        "dn4_binary"
    ]
    .value_counts()
    .sort_index()
    .rename(
        index={
            0: "DN4 < 4",
            1: "DN4 >= 4"
        }
    )
)

print(
    "\nDN4 score:"
)

display(
    patient_dn4_labels[
        "dn4_score"
    ].describe()
)

# Distribuzione turn-level
# solo informativa
print(
    "\n===== DISTRIBUZIONE TURN-LEVEL ====="
)

display(
    turn_dn4[
        "dn4_binary"
    ]
    .value_counts()
    .sort_index()
)

Turni con DN4: 3710
Pazienti con DN4: 90

===== DISTRIBUZIONE PATIENT-LEVEL =====


dn4_binary
DN4 < 4     44
DN4 >= 4    46
Name: count, dtype: int64


DN4 score:


count    90.000000
mean      3.500000
std       2.757014
min       0.000000
25%       1.000000
50%       4.000000
75%       5.750000
max       9.000000
Name: dn4_score, dtype: float64


===== DISTRIBUZIONE TURN-LEVEL =====


dn4_binary
0    1707
1    2003
Name: count, dtype: int64

20.5 — Mapping alle righe originali degli embedding

In [5]:
metadata_indexed = (
    metadata_df
    .reset_index()
    .rename(
        columns={
            "index": "embedding_row"
        }
    )
)

turn_dn4 = (
    metadata_indexed
    .merge(
        dn4_patient,
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )
    .copy()
)

turn_dn4[
    "dn4_binary"
] = (
    turn_dn4[
        "dn4_score"
    ] >= 4
).astype(int)

embedding_rows_dn4 = (
    turn_dn4[
        "embedding_row"
    ]
    .to_numpy(
        dtype=int
    )
)

print(
    "Righe selezionate:",
    len(embedding_rows_dn4)
)

print(
    "Min indice:",
    embedding_rows_dn4.min()
)

print(
    "Max indice:",
    embedding_rows_dn4.max()
)

print(
    "Indici unici:",
    len(
        np.unique(
            embedding_rows_dn4
        )
    )
)

Righe selezionate: 3710
Min indice: 0
Max indice: 3709
Indici unici: 3710


20.6 — Caricamento rappresentazioni

In [6]:
from sklearn.preprocessing import normalize

W2V_DIR = (
    RESULTS_DIR
    / "embedding_wav2vec2_turn_level"
)

# Wav2Vec2
X_wav2vec2 = np.load(
    W2V_DIR
    / "embedding_audio_wav2vec2_base_turn_level.npy"
)

X_wav2vec2 = normalize(
    X_wav2vec2,
    norm="l2"
).astype(np.float32)

# Whisper-small
X_whisper = np.load(
    EMBEDDING_DIR
    / "embedding_audio_whisper_small_l2.npy"
).astype(np.float32)

# Testo
X_text = np.load(
    EMBEDDING_DIR
    / "embedding_testuali_multilingual_minilm_l2.npy"
).astype(np.float32)

# Early fusion
X_fusion = np.load(
    EMBEDDING_DIR
    / "embedding_fusion_audio_testo_early_equal_weight.npy"
).astype(np.float32)

# Duration-only baseline
X_duration = (
    np.log1p(
        metadata_df[
            "turn_duration_seconds"
        ].to_numpy()
    )
    .reshape(-1, 1)
    .astype(np.float32)
)

REPRESENTATIONS = {
    "wav2vec2":
        X_wav2vec2,

    "whisper":
        X_whisper,

    "text":
        X_text,

    "fusion":
        X_fusion,

    "duration_only":
        X_duration
}

for name, X in REPRESENTATIONS.items():
    print(
        f"{name:14s} | "
        f"shape={X.shape} | "
        f"NaN={np.isnan(X).sum()} | "
        f"Inf={np.isinf(X).sum()}"
    )

wav2vec2       | shape=(3710, 768) | NaN=0 | Inf=0
whisper        | shape=(3710, 768) | NaN=0 | Inf=0
text           | shape=(3710, 384) | NaN=0 | Inf=0
fusion         | shape=(3710, 1152) | NaN=0 | Inf=0
duration_only  | shape=(3710, 1) | NaN=0 | Inf=0


20.7 — Stratified 5-fold a livello paziente

In [7]:
from sklearn.model_selection import StratifiedKFold

patient_table = (
    turn_dn4[
        [
            "patient_id",
            "dn4_binary"
        ]
    ]
    .drop_duplicates(
        subset=["patient_id"]
    )
    .sort_values("patient_id")
    .reset_index(drop=True)
)

patient_ids = (
    patient_table[
        "patient_id"
    ].to_numpy()
)

patient_labels = (
    patient_table[
        "dn4_binary"
    ].to_numpy()
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

patient_folds = []


for fold, (
    train_patient_idx,
    test_patient_idx
) in enumerate(
    skf.split(
        patient_ids,
        patient_labels
    ),
    start=1
):
    train_patients = patient_ids[
        train_patient_idx
    ]

    test_patients = patient_ids[
        test_patient_idx
    ]

    patient_folds.append(
        (
            train_patients,
            test_patients
        )
    )

    train_labels_fold = (
        patient_labels[
            train_patient_idx
        ]
    )

    test_labels_fold = (
        patient_labels[
            test_patient_idx
        ]
    )


    print(
        f"Fold {fold} | "
        f"train patients={len(train_patients)} "
        f"(0={np.sum(train_labels_fold == 0)}, "
        f"1={np.sum(train_labels_fold == 1)}) | "
        f"test patients={len(test_patients)} "
        f"(0={np.sum(test_labels_fold == 0)}, "
        f"1={np.sum(test_labels_fold == 1)})"
    )


    assert (
        len(
            set(train_patients)
            &
            set(test_patients)
        )
        == 0
    )

Fold 1 | train patients=72 (0=35, 1=37) | test patients=18 (0=9, 1=9)
Fold 2 | train patients=72 (0=35, 1=37) | test patients=18 (0=9, 1=9)
Fold 3 | train patients=72 (0=35, 1=37) | test patients=18 (0=9, 1=9)
Fold 4 | train patients=72 (0=35, 1=37) | test patients=18 (0=9, 1=9)
Fold 5 | train patients=72 (0=36, 1=36) | test patients=18 (0=8, 1=10)


20.8 — Funzioni score-dependent

In [8]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

def patient_balanced_weights(patient_ids_turn):
    counts = (
        pd.Series(patient_ids_turn)
        .value_counts()
    )

    weights = np.array([
        1.0 / counts[p]
        for p in patient_ids_turn
    ])

    # normalizzazione: peso medio = 1
    weights = (
        weights
        * len(weights)
        / weights.sum()
    )

    return weights

def aggregate_patient_predictions(
    patient_ids_turn,
    y_true_turn,
    probabilities
):

    df = pd.DataFrame({
        "patient_id":
            patient_ids_turn,

        "y_true":
            y_true_turn,

        "probability":
            probabilities
    })

    patient_df = (
        df
        .groupby("patient_id")
        .agg(
            y_true=("y_true", "first"),
            probability=("probability", "mean"),
            n_turns=("probability", "size")
        )
        .reset_index()
    )

    patient_df[
        "y_pred"
    ] = (
        patient_df[
            "probability"
        ] >= 0.5
    ).astype(int)

    return patient_df

def compute_patient_metrics(
    patient_df
):
    y_true = (
        patient_df[
            "y_true"
        ].to_numpy()
    )

    y_pred = (
        patient_df[
            "y_pred"
        ].to_numpy()
    )

    probability = (
        patient_df[
            "probability"
        ].to_numpy()
    )

    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probability
            )
    }

20.9 — Modelli

In [9]:
def create_models():
    rf = RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    return {
        "RF": rf,
        "XGB": xgb
    }

20.10 — RF/XGB su tutte le rappresentazioni

In [10]:
import time

SCORE_DIR = (
    RESULTS_DIR
    / "score_dependent_embeddings"
)

SCORE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

embedding_rows = (
    turn_dn4[
        "embedding_row"
    ].to_numpy(dtype=int)
)

turn_patient_ids = (
    turn_dn4[
        "patient_id"
    ].to_numpy()
)

y_turn = (
    turn_dn4[
        "dn4_binary"
    ].to_numpy()
)

score_rows = []
oof_prediction_rows = []

start_total = time.time()

for representation_name, X_full in REPRESENTATIONS.items():
    print(
        "\n"
        + "=" * 70
    )

    print(
        "RAPPRESENTAZIONE:",
        representation_name.upper()
    )

    print(
        "=" * 70
    )

    X_rep = X_full[
        embedding_rows
    ]

    for fold_number, (
        train_patients,
        test_patients
    ) in enumerate(
        patient_folds,
        start=1
    ):

        train_mask = np.isin(
            turn_patient_ids,
            train_patients
        )

        test_mask = np.isin(
            turn_patient_ids,
            test_patients
        )

        X_train = X_rep[
            train_mask
        ]

        X_test = X_rep[
            test_mask
        ]

        y_train = y_turn[
            train_mask
        ]

        y_test = y_turn[
            test_mask
        ]


        train_patient_ids_turn = (
            turn_patient_ids[
                train_mask
            ]
        )

        test_patient_ids_turn = (
            turn_patient_ids[
                test_mask
            ]
        )

        # Bilanciamento per paziente
        sample_weight = (
            patient_balanced_weights(
                train_patient_ids_turn
            )
        )

        # PCA SOLO sul training
        # Duration-only non necessita PCA
        if representation_name != "duration_only":
            pca = PCA(
                n_components=0.95,
                svd_solver="full"
            )

            X_train_model = (
                pca.fit_transform(
                    X_train
                )
            )

            X_test_model = (
                pca.transform(
                    X_test
                )
            )

            n_components = (
                X_train_model.shape[1]
            )

        else:
            X_train_model = X_train
            X_test_model = X_test
            n_components = 1

        print(
            f"\nFold {fold_number} | "
            f"train turns={len(X_train_model)} | "
            f"test turns={len(X_test_model)} | "
            f"features={n_components}"
        )

        # RF + XGB
        models = create_models()

        for model_name, model in models.items():
            start_model = time.time()

            model.fit(
                X_train_model,
                y_train,
                sample_weight=sample_weight
            )

            probabilities = (
                model.predict_proba(
                    X_test_model
                )[:, 1]
            )

            patient_predictions = (
                aggregate_patient_predictions(
                    test_patient_ids_turn,
                    y_test,
                    probabilities
                )
            )

            metrics = (
                compute_patient_metrics(
                    patient_predictions
                )
            )

            score_rows.append({
                "representation":
                    representation_name,

                "model":
                    model_name,

                "fold":
                    fold_number,

                "n_components":
                    n_components,

                **metrics
            })

            patient_predictions[
                "representation"
            ] = representation_name

            patient_predictions[
                "model"
            ] = model_name

            patient_predictions[
                "fold"
            ] = fold_number

            oof_prediction_rows.append(
                patient_predictions
            )

            print(
                f"  {model_name:3s} | "
                f"BA={metrics['balanced_accuracy']:.4f} | "
                f"AUC={metrics['roc_auc']:.4f} | "
                f"F1={metrics['f1']:.4f} | "
                f"{time.time() - start_model:.1f}s"
            )

score_results_df = pd.DataFrame(
    score_rows
)

oof_predictions_df = pd.concat(
    oof_prediction_rows,
    ignore_index=True
)

print(
    "\nTempo totale:",
    round(
        (time.time() - start_total) / 60,
        1
    ),
    "minuti"
)


RAPPRESENTAZIONE: WAV2VEC2

Fold 1 | train turns=2971 | test turns=739 | features=115
  RF  | BA=0.3889 | AUC=0.2963 | F1=0.5600 | 3.5s
  XGB | BA=0.3333 | AUC=0.3704 | F1=0.4000 | 0.9s

Fold 2 | train turns=2999 | test turns=711 | features=115
  RF  | BA=0.4444 | AUC=0.4198 | F1=0.6154 | 3.9s
  XGB | BA=0.6667 | AUC=0.5309 | F1=0.7000 | 0.8s

Fold 3 | train turns=2928 | test turns=782 | features=116
  RF  | BA=0.3889 | AUC=0.3827 | F1=0.5600 | 3.4s
  XGB | BA=0.4444 | AUC=0.3580 | F1=0.5000 | 1.0s

Fold 4 | train turns=2941 | test turns=769 | features=115
  RF  | BA=0.5000 | AUC=0.4815 | F1=0.6667 | 4.0s
  XGB | BA=0.3333 | AUC=0.4444 | F1=0.3333 | 1.3s

Fold 5 | train turns=3001 | test turns=709 | features=116
  RF  | BA=0.5000 | AUC=0.4875 | F1=0.7143 | 6.5s
  XGB | BA=0.4625 | AUC=0.5750 | F1=0.6400 | 1.9s

RAPPRESENTAZIONE: WHISPER

Fold 1 | train turns=2971 | test turns=739 | features=272
  RF  | BA=0.5000 | AUC=0.5062 | F1=0.6667 | 8.9s
  XGB | BA=0.6111 | AUC=0.5185 | F1=0.666

20.11 — Riepilogo 5-fold

In [11]:
score_summary_df = (
    score_results_df
    .groupby(
        [
            "representation",
            "model"
        ]
    )
    .agg(
        accuracy_mean=(
            "accuracy",
            "mean"
        ),

        accuracy_std=(
            "accuracy",
            "std"
        ),

        balanced_accuracy_mean=(
            "balanced_accuracy",
            "mean"
        ),

        balanced_accuracy_std=(
            "balanced_accuracy",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),

        roc_auc_mean=(
            "roc_auc",
            "mean"
        ),

        roc_auc_std=(
            "roc_auc",
            "std"
        )
    )
    .reset_index()
    .sort_values(
        "roc_auc_mean",
        ascending=False
    )
)

display(
    score_summary_df.round(4)
)

,representation,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
1,duration_only,XGB,0.6111,0.0878,0.6128,0.0892,0.6957,0.0502,0.7475,0.0441
0,duration_only,RF,0.6333,0.1083,0.6303,0.1033,0.7118,0.0910,0.6857,0.1366
5,text,XGB,0.5889,0.0843,0.5842,0.0834,0.6964,0.0433,0.5643,0.0773
8,whisper,RF,0.5111,0.0248,0.5000,0.0000,0.6762,0.0213,0.5174,0.0226
2,fusion,RF,0.5111,0.0248,0.5000,0.0000,0.6762,0.0213,0.5123,0.1429
9,whisper,XGB,0.5333,0.0633,0.5272,0.0621,0.6194,0.0812,0.5075,0.0522
4,text,RF,0.5222,0.0304,0.5161,0.0246,0.6718,0.0115,0.5050,0.1473
7,wav2vec2,XGB,0.4556,0.1383,0.4481,0.1363,0.5147,0.1552,0.4557,0.0960
3,fusion,XGB,0.4333,0.0824,0.4294,0.0822,0.5498,0.0520,0.4528,0.0948
6,wav2vec2,RF,0.4556,0.0724,0.4444,0.0556,0.6233,0.0675,0.4135,0.0788


20.12 — Performance OOF complessiva sui 90 pazienti

In [12]:
oof_summary_rows = []

for (
    representation,
    model_name
), group in oof_predictions_df.groupby(
    [
        "representation",
        "model"
    ]
):

    assert (
        group["patient_id"].nunique()
        == 90
    )

    assert (
        len(group)
        == 90
    )

    metrics = (
        compute_patient_metrics(
            group
        )
    )

    oof_summary_rows.append({
        "representation":
            representation,

        "model":
            model_name,

        **metrics
    })

oof_summary_df = (
    pd.DataFrame(
        oof_summary_rows
    )
    .sort_values(
        "roc_auc",
        ascending=False
    )
)

display(
    oof_summary_df.round(4)
)

,representation,model,accuracy,balanced_accuracy,f1,roc_auc
1,duration_only,XGB,0.6111,0.6052,0.6957,0.7134
0,duration_only,RF,0.6333,0.6275,0.7130,0.6695
5,text,XGB,0.5889,0.5815,0.6942,0.5543
8,whisper,RF,0.5111,0.5000,0.6765,0.5109
2,fusion,RF,0.5111,0.5000,0.6765,0.5104
9,whisper,XGB,0.5333,0.5282,0.6250,0.4975
4,text,RF,0.5222,0.5124,0.6718,0.4807
3,fusion,XGB,0.4333,0.4279,0.5487,0.4521
7,wav2vec2,XGB,0.4556,0.4526,0.5243,0.4481
6,wav2vec2,RF,0.4556,0.4457,0.6260,0.4224


20.13 — Audit patient-level della durata

In [13]:
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

patient_duration_df = (
    turn_dn4
    .groupby("patient_id")
    .agg(
        dn4_score=("dn4_score", "first"),
        dn4_binary=("dn4_binary", "first"),
        n_turns=("turn_duration_seconds", "size"),
        duration_mean=("turn_duration_seconds", "mean"),
        duration_median=("turn_duration_seconds", "median"),
        duration_std=("turn_duration_seconds", "std"),
        duration_total=("turn_duration_seconds", "sum")
    )
    .reset_index()
)

print("Pazienti:", len(patient_duration_df))

print("\n===== DURATA PER CLASSE DN4 =====")

display(
    patient_duration_df
    .groupby("dn4_binary")[
        [
            "n_turns",
            "duration_mean",
            "duration_median",
            "duration_total"
        ]
    ]
    .agg(["mean", "median", "std"])
    .round(3)
)

# Mann-Whitney sulle mediane patient-level
duration_0 = (
    patient_duration_df.loc[
        patient_duration_df["dn4_binary"] == 0,
        "duration_median"
    ]
    .to_numpy()
)

duration_1 = (
    patient_duration_df.loc[
        patient_duration_df["dn4_binary"] == 1,
        "duration_median"
    ]
    .to_numpy()
)

mw_stat, mw_p = mannwhitneyu(
    duration_0,
    duration_1,
    alternative="two-sided"
)

print(
    "\nMann-Whitney durata mediana:"
)

print(
    "U =",
    round(mw_stat, 3),
    "| p =",
    f"{mw_p:.4g}"
)

# AUC grezza della durata mediana
# L'orientamento può essere < 0.5, quindi mostriamo anche l'AUC orientata.

raw_auc = roc_auc_score(
    patient_duration_df["dn4_binary"],
    patient_duration_df["duration_median"]
)

oriented_auc = max(
    raw_auc,
    1 - raw_auc
)

print(
    "\nAUC grezza duration_median:",
    round(raw_auc, 4)
)

print(
    "AUC orientata:",
    round(oriented_auc, 4)
)

Pazienti: 90

===== DURATA PER CLASSE DN4 =====


n_turns                duration_mean               duration_median  \
              mean median     std          mean median    std            mean   
dn4_binary                                                                      
0           38.795   38.5  17.529         2.684  2.462  1.351           1.786   
1           43.543   40.5  17.589         2.968  2.579  1.447           1.902   

                         duration_total                    
           median    std           mean   median      std  
dn4_binary                                                 
0           1.599  0.836        110.444   79.446   87.978  
1           1.717  0.792        135.471  115.240  103.581


Mann-Whitney durata mediana:
U = 891.5 | p = 0.3327

AUC grezza duration_median: 0.5595
AUC orientata: 0.5595


20.14 — Numero di turni vs DN4

In [ ]:
turns_0 = (
    patient_duration_df.loc[
        patient_duration_df["dn4_binary"] == 0,
        "n_turns"
    ]
    .to_numpy()
)

turns_1 = (
    patient_duration_df.loc[
        patient_duration_df["dn4_binary"] == 1,
        "n_turns"
    ]
    .to_numpy()
)

u_turns, p_turns = mannwhitneyu(
    turns_0,
    turns_1,
    alternative="two-sided"
)

print(
    "Mann-Whitney numero turni:"
)

print(
    "U =",
    round(u_turns, 3),
    "| p =",
    f"{p_turns:.4g}"
)

display(
    patient_duration_df
    .groupby("dn4_binary")["n_turns"]
    .describe()
    .round(2)
)

Mann-Whitney numero turni:
U = 889.0 | p = 0.3225


,count,mean,std,min,25%,50%,75%,max
dn4_binary,,,,,,,,
0,44.0,38.80,17.53,8.0,28.75,38.5,47.25,97.0
1,46.0,43.54,17.59,17.0,32.00,40.5,52.00,90.0


20.15 — Salvataggio risultati RF/XGB

In [15]:
score_results_df.to_csv(
    SCORE_DIR / "risultati_5fold_RF_XGB_DN4.csv",
    index=False
)

score_summary_df.to_csv(
    SCORE_DIR / "riepilogo_5fold_RF_XGB_DN4.csv",
    index=False
)

oof_predictions_df.to_csv(
    SCORE_DIR / "predizioni_OOF_RF_XGB_DN4.csv",
    index=False
)

oof_summary_df.to_csv(
    SCORE_DIR / "riepilogo_OOF_RF_XGB_DN4.csv",
    index=False
)

patient_duration_df.to_csv(
    SCORE_DIR / "audit_durata_patient_level_DN4.csv",
    index=False
)

print("✓ Risultati salvati.")

✓ Risultati salvati.


20.16 — Permutation test patient-level
- Duration-only + XGBoost

In [16]:
from pathlib import Path
import numpy as np
import pandas as pd
import time

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score
)

N_PERMUTATIONS = 1000

PERM_PATH = (
    SCORE_DIR
    / "permutation_duration_only_xgb_DN4.npy"
)

# Dati duration-only allineati a turn_dn4
X_duration_dn4 = (
    X_duration[
        embedding_rows
    ]
)

# Risultato osservato
observed_oof = (
    oof_predictions_df[
        (
            oof_predictions_df["representation"]
            == "duration_only"
        )
        &
        (
            oof_predictions_df["model"]
            == "XGB"
        )
    ]
    .sort_values("patient_id")
    .copy()
)

OBS_AUC = roc_auc_score(
    observed_oof["y_true"],
    observed_oof["probability"]
)

OBS_BA = balanced_accuracy_score(
    observed_oof["y_true"],
    observed_oof["y_pred"]
)

print(
    "AUC osservata:",
    round(OBS_AUC, 4)
)

print(
    "BA osservata:",
    round(OBS_BA, 4)
)

# Checkpoint / resume
# colonne:
# 0 = AUC
# 1 = Balanced Accuracy

if PERM_PATH.exists():
    permutation_results = np.load(
        PERM_PATH
    )

    if permutation_results.shape != (
        N_PERMUTATIONS,
        2
    ):
        raise ValueError(
            "Shape checkpoint permutation inattesa."
        )

    print(
        "Checkpoint trovato:",
        np.isfinite(
            permutation_results[:, 0]
        ).sum(),
        "permutazioni già completate"
    )
else:
    permutation_results = np.full(
        (
            N_PERMUTATIONS,
            2
        ),
        np.nan,
        dtype=float
    )

start_time = time.time()

for perm_idx in range(
    N_PERMUTATIONS
):
    # già eseguita
    if np.isfinite(
        permutation_results[
            perm_idx,
            0
        ]
    ):
        continue

    # Permutazione deterministica a livello paziente
    rng = np.random.default_rng(
        10000 + perm_idx
    )

    permuted_patient_labels = (
        rng.permutation(
            patient_labels
        )
    )

    perm_label_map = dict(
        zip(
            patient_ids,
            permuted_patient_labels
        )
    )

    y_perm_turn = np.asarray(
        [
            perm_label_map[p]
            for p in turn_patient_ids
        ],
        dtype=int
    )


    perm_oof_rows = []


    # Stessi 5 fold patient-level
    for (
        train_patients,
        test_patients
    ) in patient_folds:

        train_mask = np.isin(
            turn_patient_ids,
            train_patients
        )

        test_mask = np.isin(
            turn_patient_ids,
            test_patients
        )

        X_train = X_duration_dn4[
            train_mask
        ]

        X_test = X_duration_dn4[
            test_mask
        ]

        y_train = y_perm_turn[
            train_mask
        ]

        y_test = y_perm_turn[
            test_mask
        ]

        train_patient_ids_turn = (
            turn_patient_ids[
                train_mask
            ]
        )

        test_patient_ids_turn = (
            turn_patient_ids[
                test_mask
            ]
        )

        weights = (
            patient_balanced_weights(
                train_patient_ids_turn
            )
        )

        model = create_models()[
            "XGB"
        ]


        model.fit(
            X_train,
            y_train,
            sample_weight=weights
        )

        probabilities = (
            model.predict_proba(
                X_test
            )[:, 1]
        )

        patient_predictions = (
            aggregate_patient_predictions(
                test_patient_ids_turn,
                y_test,
                probabilities
            )
        )

        perm_oof_rows.append(
            patient_predictions
        )

    perm_oof = pd.concat(
        perm_oof_rows,
        ignore_index=True
    )

    perm_auc = roc_auc_score(
        perm_oof["y_true"],
        perm_oof["probability"]
    )

    perm_ba = balanced_accuracy_score(
        perm_oof["y_true"],
        perm_oof["y_pred"]
    )

    permutation_results[
        perm_idx
    ] = [
        perm_auc,
        perm_ba
    ]

    if (
        (perm_idx + 1) % 100 == 0
    ):
        np.save(
            PERM_PATH,
            permutation_results
        )

        print(
            f"{perm_idx + 1}/{N_PERMUTATIONS} completate"
        )

np.save(
    PERM_PATH,
    permutation_results
)

print(
    "\nTempo:",
    round(
        (time.time() - start_time) / 60,
        1
    ),
    "minuti"
)

AUC osservata: 0.7134
BA osservata: 0.6052
100/1000 completate
200/1000 completate
300/1000 completate
400/1000 completate
500/1000 completate
600/1000 completate
700/1000 completate
800/1000 completate
900/1000 completate
1000/1000 completate

Tempo: 13.8 minuti


20.17 — Risultato permutation test

In [17]:
null_auc = (
    permutation_results[:, 0]
)

null_ba = (
    permutation_results[:, 1]
)

p_auc = (
    1
    +
    np.sum(
        null_auc >= OBS_AUC
    )
) / (
    len(null_auc) + 1
)

p_ba = (
    1
    +
    np.sum(
        null_ba >= OBS_BA
    )
) / (
    len(null_ba) + 1
)

print("===== ROC-AUC =====")

print(
    "Osservata:",
    round(OBS_AUC, 4)
)

print(
    "Null mean:",
    round(null_auc.mean(), 4)
)

print(
    "Null std:",
    round(null_auc.std(), 4)
)

print(
    "95° percentile:",
    round(
        np.percentile(
            null_auc,
            95
        ),
        4
    )
)

print(
    "p-value:",
    round(p_auc, 4)
)

print("\n===== BALANCED ACCURACY =====")

print(
    "Osservata:",
    round(OBS_BA, 4)
)

print(
    "Null mean:",
    round(null_ba.mean(), 4)
)

print(
    "Null std:",
    round(null_ba.std(), 4)
)

print(
    "95° percentile:",
    round(
        np.percentile(
            null_ba,
            95
        ),
        4
    )
)

print(
    "p-value:",
    round(p_ba, 4)
)

===== ROC-AUC =====
Osservata: 0.7134
Null mean: 0.4091
Null std: 0.0677
95° percentile: 0.5292
p-value: 0.001

===== BALANCED ACCURACY =====
Osservata: 0.6052
Null mean: 0.4337
Null std: 0.0542
95° percentile: 0.5296
p-value: 0.002


20.18 — Permutation test CORRETTO
- Duration-only + XGBoost
- Ricostruzione dei fold stratificati ad ogni permutazione

In [18]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score
)

N_PERMUTATIONS = 1000

PERM_CORRECTED_PATH = (
    SCORE_DIR
    / "permutation_duration_only_xgb_DN4_CORRECTED.npy"
)

X_duration_dn4 = (
    X_duration[
        embedding_rows
    ]
)

print(
    "AUC osservata:",
    round(OBS_AUC, 4)
)

print(
    "BA osservata:",
    round(OBS_BA, 4)
)

# Nuovo checkpoint
if PERM_CORRECTED_PATH.exists():
    permutation_corrected = np.load(
        PERM_CORRECTED_PATH
    )
    print(
        "Checkpoint trovato:",
        np.isfinite(
            permutation_corrected[:, 0]
        ).sum(),
        "permutazioni"
    )
else:
    permutation_corrected = np.full(
        (
            N_PERMUTATIONS,
            2
        ),
        np.nan,
        dtype=float
    )

start_time = time.time()

for perm_idx in range(
    N_PERMUTATIONS
):
    if np.isfinite(
        permutation_corrected[
            perm_idx,
            0
        ]
    ):
        continue
   
    # Permutazione delle label tra i 90 pazienti
    rng = np.random.default_rng(
        20000 + perm_idx
    )

    permuted_patient_labels = (
        rng.permutation(
            patient_labels
        )
    )

    perm_label_map = dict(
        zip(
            patient_ids,
            permuted_patient_labels
        )
    )

    y_perm_turn = np.asarray(
        [
            perm_label_map[p]
            for p in turn_patient_ids
        ],
        dtype=int
    )

    # IMPORTANTE:nuovi fold stratificati sulle label permutate
    skf_perm = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    perm_oof_rows = []

    for (
        train_patient_idx,
        test_patient_idx
    ) in skf_perm.split(
        patient_ids,
        permuted_patient_labels
    ):

        train_patients = (
            patient_ids[
                train_patient_idx
            ]
        )

        test_patients = (
            patient_ids[
                test_patient_idx
            ]
        )

        train_mask = np.isin(
            turn_patient_ids,
            train_patients
        )

        test_mask = np.isin(
            turn_patient_ids,
            test_patients
        )

        X_train = X_duration_dn4[
            train_mask
        ]

        X_test = X_duration_dn4[
            test_mask
        ]

        y_train = y_perm_turn[
            train_mask
        ]

        y_test = y_perm_turn[
            test_mask
        ]

        train_patient_ids_turn = (
            turn_patient_ids[
                train_mask
            ]
        )

        test_patient_ids_turn = (
            turn_patient_ids[
                test_mask
            ]
        )

        weights = (
            patient_balanced_weights(
                train_patient_ids_turn
            )
        )

        model = create_models()[
            "XGB"
        ]

        model.fit(
            X_train,
            y_train,
            sample_weight=weights
        )

        probabilities = (
            model.predict_proba(
                X_test
            )[:, 1]
        )

        patient_predictions = (
            aggregate_patient_predictions(
                test_patient_ids_turn,
                y_test,
                probabilities
            )
        )

        perm_oof_rows.append(
            patient_predictions
        )

    perm_oof = pd.concat(
        perm_oof_rows,
        ignore_index=True
    )

    # Ogni paziente deve comparire una sola volta
    assert len(perm_oof) == 90
    assert perm_oof["patient_id"].nunique() == 90

    perm_auc = roc_auc_score(
        perm_oof["y_true"],
        perm_oof["probability"]
    )

    perm_ba = balanced_accuracy_score(
        perm_oof["y_true"],
        perm_oof["y_pred"]
    )

    permutation_corrected[
        perm_idx
    ] = [
        perm_auc,
        perm_ba
    ]

    if (
        (perm_idx + 1) % 100 == 0
    ):
        np.save(
            PERM_CORRECTED_PATH,
            permutation_corrected
        )

        print(
            f"{perm_idx + 1}/"
            f"{N_PERMUTATIONS} completate"
        )

np.save(
    PERM_CORRECTED_PATH,
    permutation_corrected
)

print(
    "\nTempo:",
    round(
        (
            time.time()
            - start_time
        ) / 60,
        1
    ),
    "minuti"
)

AUC osservata: 0.7134
BA osservata: 0.6052
100/1000 completate
200/1000 completate
300/1000 completate
400/1000 completate
500/1000 completate
600/1000 completate
700/1000 completate
800/1000 completate
900/1000 completate
1000/1000 completate

Tempo: 12.3 minuti


20.19 — Risultato permutation test corretto

In [19]:
null_auc_corrected = (
    permutation_corrected[:, 0]
)

null_ba_corrected = (
    permutation_corrected[:, 1]
)

p_auc_corrected = (
    1
    +
    np.sum(
        null_auc_corrected
        >= OBS_AUC
    )
) / (
    N_PERMUTATIONS + 1
)

p_ba_corrected = (
    1
    +
    np.sum(
        null_ba_corrected
        >= OBS_BA
    )
) / (
    N_PERMUTATIONS + 1
)

print("===== ROC-AUC =====")

print(
    "Osservata:",
    round(OBS_AUC, 4)
)

print(
    "Null mean:",
    round(
        null_auc_corrected.mean(),
        4
    )
)

print(
    "Null std:",
    round(
        null_auc_corrected.std(),
        4
    )
)

print(
    "95° percentile:",
    round(
        np.percentile(
            null_auc_corrected,
            95
        ),
        4
    )
)

print(
    "p-value:",
    round(
        p_auc_corrected,
        4
    )
)

print("\n===== BALANCED ACCURACY =====")

print(
    "Osservata:",
    round(OBS_BA, 4)
)

print(
    "Null mean:",
    round(
        null_ba_corrected.mean(),
        4
    )
)

print(
    "Null std:",
    round(
        null_ba_corrected.std(),
        4
    )
)

print(
    "95° percentile:",
    round(
        np.percentile(
            null_ba_corrected,
            95
        ),
        4
    )
)

print(
    "p-value:",
    round(
        p_ba_corrected,
        4
    )
)

===== ROC-AUC =====
Osservata: 0.7134
Null mean: 0.4867
Null std: 0.0773
95° percentile: 0.6141
p-value: 0.001

===== BALANCED ACCURACY =====
Osservata: 0.6052
Null mean: 0.4938
Null std: 0.0549
95° percentile: 0.5845
p-value: 0.024


20.20 — Salvataggio finale permutation test

In [20]:
permutation_summary_df = pd.DataFrame(
    {
        "metric": [
            "roc_auc",
            "balanced_accuracy"
        ],

        "observed": [
            OBS_AUC,
            OBS_BA
        ],

        "null_mean": [
            null_auc_corrected.mean(),
            null_ba_corrected.mean()
        ],

        "null_std": [
            null_auc_corrected.std(),
            null_ba_corrected.std()
        ],

        "null_95_percentile": [
            np.percentile(
                null_auc_corrected,
                95
            ),
            np.percentile(
                null_ba_corrected,
                95
            )
        ],

        "p_value": [
            p_auc_corrected,
            p_ba_corrected
        ]
    }
)

permutation_summary_df.to_csv(
    SCORE_DIR
    / "permutation_test_CORRECTED_duration_only_XGB_DN4.csv",
    index=False
)

print("✓ Permutation test corretto salvato.")

display(
    permutation_summary_df.round(4)
)

✓ Permutation test corretto salvato.


,metric,observed,null_mean,null_std,null_95_percentile,p_value
0,roc_auc,0.7134,0.4867,0.0773,0.6141,0.001
1,balanced_accuracy,0.6052,0.4938,0.0549,0.5845,0.024
